# Assignment 1

## Due on Wednesday 4th February at 12:00pm
### Submit your jupyter notebook on QMPlus (tab Assessments) with name LAST_NAME_Assignment_1.ipynb

## Naive Bayes classifier on Habitable Worlds catalogue

### The goal is to train a classifier that estimates the habitability of a planet.
In principle, this is a multi class problem, with P_HABITABLE  = 0 (inhabitable) or 1 (conservatively habitable) or 2 (optimistically habitable). However:

For now, we will treat is a binary classification. <b>Change P_HABITABLE  = 1 or 2 to P_HABITABLE = 1 </b>

The definition of (most of) the columns can be found here:
https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html#columns

### Data
Use the file in ~/teaching_material/data/hwc_assignment_train.csv to train your model.

Once trained, use your model to classify the planets in  ~/teaching_material/data/hwc_assignment_test.csv and report your metrics.

### Hints
The dataset contains both numeric and not-numeric features. You can choose to discard the non-numeric features, or to translate them into descrete features. It also contains some features with many empty (NaN) values. You can choose to impute or discard the missing features. 
Remember that GaussianNB from sklearn.naive_bayes is not appropriate for categorical features. Use MultinomialNB instead.

### Grading 
This is a *formative assessment* and it is NOT graded. 

However, we will use as a score:

Score = ROC-AUC Score * Precision, as calculated by sklearn.metrics

Please report the Score (on the test set) at the end of the notebook.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



In [131]:
#Score Function
def print_metrics(y, y_pred, y_prob):
    # 1. Accuracy
    accuracy = accuracy_score(y, y_pred)
    print(f"Accuracy: {accuracy:.2f}")

    # 2. Precision
    precision = precision_score(y, y_pred)
    print(f"Precision: {precision:.2f}")

    # 3. Recall (Sensitivity)
    recall = recall_score(y, y_pred)
    print(f"Recall (Sensitivity): {recall:.2f}")

    # 4. F1 Score
    f1 = f1_score(y, y_pred)
    print(f"F1 Score: {f1:.2f}")

    # 5. ROC-AUC Score
    roc_auc = roc_auc_score(y, y_prob)
    print(f"ROC-AUC Score: {roc_auc:.2f}")

    # 6. Confusion Matrix
    conf_matrix = confusion_matrix(y, y_pred)
    print("Confusion Matrix:")
    print(conf_matrix)

    score = roc_auc*accuracy*100

    print(f"Score =  {score:.2f}")
    
    return score

In [ ]:
#Path
TRAIN_PATH = "data/hwc_assignment_train.csv"
TEST_PATH = "data/hwc_assignment_test.csv"

#Columns
TARGET_COLUMN = "P_HABITABLE"
FEATURE_COLUMNS = ['P_MASS','P_RADIUS','P_SEMI_MAJOR_AXIS','P_ECCENTRICITY','S_TEMPERATURE','S_MASS','S_RADIUS','S_METALLICITY','S_AGE','S_LOG_LUM','P_ESCAPE','P_POTENTIAL','P_GRAVITY','P_DENSITY','P_HILL_SPHERE','P_DISTANCE','P_PERIASTRON','P_APASTRON','P_DISTANCE_EFF','P_FLUX','P_TEMP_EQUIL','S_LUMINOSITY','S_SNOW_LINE','S_ABIO_ZONE','S_TIDAL_LOCK','P_HABITABLE']

#Hyperparameters
RANDOM_STATE = 2003
VAL_SIZE = 0.05
CLASS_WEIGHT = 'balanced'
LR = 0.05
MAX_DEPTH = 6
MAX_LEAF_NODES = 32
MIN_SAMPLES_LEAF = 20
EARLY_STOPPING = True

def build_pipeline():
    model = HistGradientBoostingClassifier(learning_rate=LR,max_depth=MAX_DEPTH, max_leaf_nodes=MAX_LEAF_NODES, min_samples_leaf=MIN_SAMPLES_LEAF, early_stopping=EARLY_STOPPING, random_state=RANDOM_STATE, class_weight=CLASS_WEIGHT)
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ("model", model)
    ])
    return pipeline

#Function to load and check dataset
def check_data(dataset):
    inf_counts = np.isinf(dataset).sum()
    nan_counts = dataset.isna().sum()

    summary_df = pd.DataFrame({'Inf Count': inf_counts,'NaN Count': nan_counts})

    zero_counts = (dataset == 0).sum()
    print(f"Number of zero values in each column:\n{zero_counts}")

    num_rows_with_nan = dataset.isna().any(axis=1).sum()
    print(f"Number of rows with at least one NaN: {num_rows_with_nan}")

    problematic_col = summary_df[(summary_df['Inf Count'] > 0) | (summary_df['NaN Count'] > 0)]
    print(f"Shape of dataset: {dataset.shape}")
    print(f"Total infs: {inf_counts.sum()}")
    print(f"Total NaNs: {nan_counts.sum()}")

    if not problematic_col.empty:
        print("Columns with issues:")
        print(problematic_col)
    else:
        print("No inf or NaN values found in the dataset.")
        print(problematic_col.sort_values(["Inf Count","NaN Count"], ascending=False))

    return summary_df

#Function to load dataset
def prepare_dataset():
    train_data = pd.read_csv(TRAIN_PATH)
    test_data = pd.read_csv(TEST_PATH)

    train_data.columns = train_data.columns.str.strip()
    test_data.columns = test_data.columns.str.strip()

    # Select only relevant features
    train_data = train_data[FEATURE_COLUMNS]
    test_data = test_data[FEATURE_COLUMNS]

    # Convert target variable to binary
    train_data[TARGET_COLUMN] = (train_data[TARGET_COLUMN] != 0).astype(int)
    test_data[TARGET_COLUMN]  = (test_data[TARGET_COLUMN] != 0).astype(int)

    # Split features and target variable
    x_train = train_data.drop(columns=[TARGET_COLUMN])
    y_train = train_data[TARGET_COLUMN]

    x_test = test_data.drop(columns=[TARGET_COLUMN])
    y_test = test_data[TARGET_COLUMN]

    x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=VAL_SIZE, random_state=RANDOM_STATE, shuffle=True, stratify=y_train)
    
    return x_train, y_train, x_val, y_val, x_test, y_test

def main():
    x_train, y_train, x_val, y_val, x_test, y_test = prepare_dataset()
    print("Datasets prepared successfully.")

    pipeline = build_pipeline()
    pipeline.fit(x_train, y_train)
    print("Model training completed.")
    val_probs = pipeline.predict_proba(x_val)[:, 1]
    val_pred = (val_probs >= 0.5).astype(int)

    print("Validation Metrics:")
    print("Validation ROC-AUC Score:", roc_auc_score(y_val, val_probs))
    print("Validation PR-AUC:", average_precision_score(y_val, val_probs))
    print(classification_report(y_val, val_pred, digits=3))

    test_probs = pipeline.predict_proba(x_test)[:, 1]
    test_pred = (test_probs >= 0.5).astype(int)

    print("Test Metrics:")
    print("Test ROC-AUC Score:", roc_auc_score(y_test, test_probs))
    print("Test PR-AUC:", average_precision_score(y_test, test_probs))
    print(classification_report(y_test, test_pred, digits=3))

    print_metrics(y_test, test_pred, test_probs)

main()

Datasets prepared successfully.
Model training completed.
Validation Metrics:
Validation ROC-AUC Score: 1.0
Validation PR-AUC: 1.0
              precision    recall  f1-score   support

           0      1.000     1.000     1.000       200
           1      1.000     1.000     1.000         2

    accuracy                          1.000       202
   macro avg      1.000     1.000     1.000       202
weighted avg      1.000     1.000     1.000       202

Test Metrics:
Test ROC-AUC Score: 0.9993158182813355
Test PR-AUC: 0.9613155363155363
              precision    recall  f1-score   support

           0      0.997     0.999     0.998      1392
           1      0.944     0.810     0.872        21

    accuracy                          0.996      1413
   macro avg      0.971     0.904     0.935      1413
weighted avg      0.996     0.996     0.996      1413

Accuracy: 1.00
Precision: 0.94
Recall (Sensitivity): 0.81
F1 Score: 0.87
ROC-AUC Score: 1.00
Confusion Matrix:
[[1391    1]
 [   4

TEST SCORE 99.58